# Appendix H — Deterministic factor reconstruction from a legacy 10-year stream

## Why a portfolio manager should care

The historical AI macro layer does not directly set the portfolio. It first characterizes the macro regime on five bounded axes, then flows through a fixed exposure map, recall-risk attenuation, and a Black–Litterman (BL) view input. This appendix makes that chain visible so an investment professional can separate:

1. **recorded model behavior** — the five macro-regime loadings;
2. **the deterministic translation** — loading → unitless exposure tilt → recall-adjusted tilt → BL `Q`; and
3. **observed portfolio output** — persisted target weights and equity levels.

This is a presentation and integrity audit. It does not infer performance attribution, rebuild HRP or BL, create a return forecast, or recommend an allocation.

## Source status and limits

- **Non-manifested legacy/reconstructed source:** `data/streams/10y/`; it is not a completed `factor_run.v1` bundle and has no `manifest.json` / `COMPLETED` contract.
- **PIT meaning here:** every macro row is selected strictly before its factor rebalance date. This is timestamp/date-as-of control, **not release-vintage PIT**: macro release timestamps and historical vintages are unavailable.
- The five AI loadings are continuous values in `[-1, +1]` that characterize a macro regime. They are **not** forecasts, expected returns, or trade instructions.
- A signed exposure tilt is unitless. The shown `Q` is the deterministic BL view input `adjusted_tilt × conviction / 252`, **not a return forecast**.
- The legacy stream does not include HRP-only or BL-only historical return/equity paths. This appendix therefore does **not** calculate or claim an AI-versus-HRP performance delta, P&L contribution, alpha, or attribution.
- The source was reconstructed historically. Results do not establish prospective forecasting skill and are not investment advice.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import struct
import sys
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from matplotlib.colors import LinearSegmentedColormap


EXPECTED_HASHES = {
    "loadings": "97b3eaff5a004be0ce0bbe3b69264da9410cf0eb5230c1cebdeb854e05d1ea83",
    "decision_log": "d0f77d59f92854965bce3cc6c543b4738dfd7c4ccb9883599d63f45928db042c",
    "targets": "c5bb7dda289377ea0d3fc4938abf19a7eea14719478611bf0201c743310a00fc",
    "equity": "6fedeb27c327580ea83fc599adadc3a6ea8c3cff54293ea138600b267e764296",
    "header": "b42c855970619c4eb2f33a1e537faa6a3a1d4c9c27bb2d2c013f287d5a07c392",
    "macro_panel": "1444e80a3e1d9e375e581d829c507b84e406e504dfed421e072edaa6eb061cac",
}
SOURCE_STATUS = "non-manifested legacy/reconstructed"
AXES = ("inflation", "growth", "credit_stress", "policy", "risk_appetite")
MACRO_COLUMNS = ("cpi_yoy_z", "t10y2y_z", "hy_oas_z")
ASSETS = ("SWDA.L", "XLK", "IAU", "BIL")
ASSET_KEYS = {"SWDA.L": "Asset_A", "XLK": "Asset_B", "IAU": "Asset_C", "BIL": "Asset_D"}
ASSET_CATEGORIES = {
    "SWDA.L": "world_equity",
    "XLK": "tech_sector",
    "IAU": "gold_commodity",
    "BIL": "short_treasury_cash",
}
EXPOSURE_MAP = {
    "world_equity": {"inflation": -0.3, "growth": 1.0, "credit_stress": -0.8, "policy": -0.3, "risk_appetite": 0.6},
    "tech_sector": {"inflation": -0.6, "growth": 1.0, "credit_stress": -0.7, "policy": -0.5, "risk_appetite": 0.8},
    "gold_commodity": {"inflation": 0.8, "growth": -0.4, "credit_stress": 0.7, "policy": 0.1, "risk_appetite": -0.5},
    "short_treasury_cash": {"inflation": 0.3, "growth": -0.6, "credit_stress": 0.6, "policy": 0.8, "risk_appetite": -0.6},
}
SOURCE_FILES = {
    "loadings": "factor_loadings_ext2026.parquet",
    "decision_log": "factor_decision_log_ext2026.json",
    "targets": "factor_targets_ext2026.parquet",
    "equity": "factor_equity_ext2026.parquet",
    "header": "factor_ext2026_run_header.json",
}
SURFACE = "#fcfcfb"
INK = "#0b0b0b"
SECONDARY = "#52514e"
GRID = "#e1e0d9"
DIVERGING_CMAP = LinearSegmentedColormap.from_list("appendix_h_diverging", ["#2166ac", "#f0efec", "#b2182b"])
ASSET_COLORS = {"SWDA.L": "#2a78d6", "XLK": "#eb6834", "IAU": "#1baf7a", "BIL": "#8e5ea2"}
ASSET_MARKERS = {"SWDA.L": "o", "XLK": "s", "IAU": "^", "BIL": "D"}
LOADING_Z_MIN_HISTORY = 12
LOADING_Z_DDOF = 1


def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / ".git").exists():
            return candidate
    raise RuntimeError("Repository root not found; run this notebook inside the repository.")


def resolve_repo_path(repo_root: Path, raw: str | Path, label: str) -> Path:
    candidate = Path(raw).expanduser()
    if not candidate.is_absolute():
        candidate = repo_root / candidate
    resolved = candidate.resolve()
    try:
        resolved.relative_to(repo_root)
    except ValueError as exc:
        raise ValueError(f"{label} must remain inside repository root {repo_root}: {resolved}") from exc
    return resolved


def repo_relative_path(repo_root: Path, path: Path, label: str) -> str:
    try:
        return path.resolve().relative_to(repo_root.resolve()).as_posix()
    except ValueError as exc:
        raise ValueError(f"{label} must remain inside repository root {repo_root}: {path}") from exc


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def require_hash(path: Path, expected: str, label: str) -> str:
    observed = sha256_file(path)
    if observed != expected:
        raise ValueError(f"{label} SHA-256 mismatch: expected={expected} observed={observed}")
    return observed


def source_snapshot(paths: dict[str, Path]) -> dict[str, str]:
    return {name: sha256_file(path) for name, path in paths.items()}


def assert_sources_unchanged(paths: dict[str, Path], before: dict[str, str]) -> None:
    if source_snapshot(paths) != before:
        raise RuntimeError("Appendix H source inputs changed during rendering")


def prior_macro_row(macro: pd.DataFrame, rebalance_date: pd.Timestamp) -> pd.Series:
    candidates = macro.loc[macro.index < rebalance_date]
    if candidates.empty:
        raise ValueError(f"No macro row strictly before rebalance date {rebalance_date.date()}")
    complete = candidates.dropna(subset=list(MACRO_COLUMNS))
    if complete.empty:
        raise ValueError(f"No complete macro z-score row strictly before rebalance date {rebalance_date.date()}")
    row = complete.iloc[-1]
    if not np.isfinite(row.loc[list(MACRO_COLUMNS)].to_numpy(dtype=float)).all():
        raise ValueError(f"Non-finite macro z-score row selected for {rebalance_date.date()}")
    return row


def equity_observation_on_or_after(equity: pd.DataFrame, rebalance_date: pd.Timestamp) -> tuple[pd.Timestamp, float]:
    position = int(equity.index.searchsorted(rebalance_date, side="left"))
    if position >= len(equity):
        raise ValueError(f"No equity observation on or after rebalance date {rebalance_date.date()}")
    observed_date = pd.Timestamp(equity.index[position])
    if observed_date < rebalance_date:
        raise ValueError("Equity observation lookup moved backward in time")
    return observed_date, float(equity.iloc[position]["value"])


def expanding_prior_zscore(values: pd.Series) -> pd.Series:
    result = pd.Series(np.nan, index=values.index, dtype=float)
    history: list[float] = []
    for date, value in values.items():
        if pd.notna(value) and len(history) >= LOADING_Z_MIN_HISTORY:
            sigma = float(np.std(history, ddof=LOADING_Z_DDOF))
            if np.isfinite(sigma) and sigma > 0.0:
                result.loc[date] = (float(value) - float(np.mean(history))) / sigma
        if pd.notna(value):
            history.append(float(value))
    return result


def png_dimensions(path: Path) -> tuple[int, int]:
    with path.open("rb") as handle:
        header = handle.read(24)
    if len(header) != 24 or header[:8] != b"\x89PNG\r\n\x1a\n":
        raise ValueError(f"Not a PNG: {path}")
    return struct.unpack(">II", header[16:24])


def output_inventory(output_dir: Path, names: tuple[str, ...]) -> dict[str, dict]:
    inventory: dict[str, dict] = {}
    for name in names:
        path = output_dir / name
        width, height = png_dimensions(path)
        inventory[name] = {
            "file": name,
            "sha256": sha256_file(path),
            "bytes": path.stat().st_size,
            "width_px": width,
            "height_px": height,
            "media_type": "image/png",
        }
    return inventory


REPO_ROOT = find_repo_root(Path.cwd())
IMPORT_REPO_ROOT = REPO_ROOT
if str(IMPORT_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(IMPORT_REPO_ROOT))
from macro_framework.appendix_presentation import markdown_lines

SOURCE_DIR = resolve_repo_path(REPO_ROOT, os.environ.get("APPENDIX_H_SOURCE_DIR", "data/streams/10y"), "APPENDIX_H_SOURCE_DIR")
MACRO_PATH = resolve_repo_path(REPO_ROOT, os.environ.get("APPENDIX_H_MACRO_PANEL", "data/macro_panel_monthly.parquet"), "APPENDIX_H_MACRO_PANEL")
NOTEBOOK_PATH = resolve_repo_path(REPO_ROOT, os.environ.get("APPENDIX_H_NOTEBOOK_PATH", "notebooks/appendix_h_deterministic_factor.ipynb"), "APPENDIX_H_NOTEBOOK_PATH")
OUTPUT_DIR = resolve_repo_path(REPO_ROOT, os.environ.get("APPENDIX_H_OUTPUT_DIR", "reports/appendix_h_deterministic_factor"), "APPENDIX_H_OUTPUT_DIR")
if OUTPUT_DIR == SOURCE_DIR or OUTPUT_DIR.is_relative_to(SOURCE_DIR) or SOURCE_DIR.is_relative_to(OUTPUT_DIR) or OUTPUT_DIR == MACRO_PATH.parent or MACRO_PATH.parent.is_relative_to(OUTPUT_DIR):
    raise ValueError("Appendix H output namespace must be separate from legacy source and macro input directories")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

source_paths = {name: SOURCE_DIR / filename for name, filename in SOURCE_FILES.items()} | {"macro_panel": MACRO_PATH}
if any(not path.is_file() for path in source_paths.values()):
    missing = [str(path) for path in source_paths.values() if not path.is_file()]
    raise FileNotFoundError(f"Appendix H required source file(s) missing: {missing}")
for name, path in source_paths.items():
    require_hash(path, EXPECTED_HASHES[name], name)
source_hashes_before = source_snapshot(source_paths)

plt.rcParams.update({
    "font.family": "sans-serif", "font.size": 9, "axes.titlesize": 12,
    "axes.labelsize": 9, "axes.edgecolor": "#c3c2b7", "axes.linewidth": 0.8,
    "axes.facecolor": SURFACE, "figure.facecolor": SURFACE, "grid.color": GRID,
    "grid.linewidth": 0.6, "grid.linestyle": "-", "legend.frameon": False,
    "lines.linewidth": 1.8, "lines.solid_capstyle": "round",
})

In [ ]:
loadings = pd.read_parquet(source_paths["loadings"]).copy()
targets = pd.read_parquet(source_paths["targets"]).copy()
equity = pd.read_parquet(source_paths["equity"]).copy()
macro = pd.read_parquet(source_paths["macro_panel"]).copy()
decision_log = json.loads(source_paths["decision_log"].read_text(encoding="utf-8"))
header = json.loads(source_paths["header"].read_text(encoding="utf-8"))

for label, frame in (("loadings", loadings), ("targets", targets), ("equity", equity), ("macro", macro)):
    if not isinstance(frame.index, pd.DatetimeIndex):
        raise ValueError(f"{label} requires a DatetimeIndex")
    if frame.index.tz is not None or not frame.index.is_monotonic_increasing or frame.index.has_duplicates:
        raise ValueError(f"{label} index must be timezone-naive, increasing, and unique")

expected_loading_columns = ["parse_ok", "segment", "variant", *AXES]
if list(loadings.columns) != expected_loading_columns:
    raise ValueError(f"Legacy loading schema changed: expected={expected_loading_columns} observed={list(loadings.columns)}")
if len(loadings) != 126 or loadings.index.min() != pd.Timestamp("2016-01-04") or loadings.index.max() != pd.Timestamp("2026-06-01"):
    raise ValueError("Legacy loading calendar must contain exactly 126 2016-01-04..2026-06-01 observations")
if set(loadings["variant"].astype(str)) != {"pit"}:
    raise ValueError("Appendix H requires only the legacy PIT loading stream")
if int(loadings["parse_ok"].sum()) != 125 or list(loadings.index[~loadings["parse_ok"]]) != [pd.Timestamp("2024-03-01")]:
    raise ValueError("Legacy loading parse status must retain the one known 2024-03-01 failure")
valid_loading_values = loadings.loc[loadings["parse_ok"], list(AXES)].to_numpy(dtype=float)
if not np.isfinite(valid_loading_values).all() or (np.abs(valid_loading_values) > 1.0).any():
    raise ValueError("Parsed legacy loadings must be finite and bounded in [-1, +1]")
if loadings.loc[~loadings["parse_ok"], list(AXES)].notna().any(axis=None):
    raise ValueError("Failed legacy loading parse must remain unavailable, not imputed")

for key in ("p_memorized", "parse_ok", "steered", "conviction", "loadings", "views"):
    if not isinstance(decision_log.get(key), dict):
        raise ValueError(f"Legacy decision log is missing mapping {key}")
log_dates = pd.DatetimeIndex(pd.to_datetime(list(decision_log["p_memorized"])))
if not log_dates.equals(loadings.index):
    raise ValueError("Decision-log calendar must equal loading-table calendar")
for key in ("parse_ok", "steered", "conviction", "loadings", "views"):
    logged_dates = pd.DatetimeIndex(pd.to_datetime(list(decision_log[key])))
    if logged_dates.difference(loadings.index).size or loadings.index.difference(logged_dates).size:
        raise ValueError(f"Decision-log key coverage mismatch for {key}")

if list(targets.columns) != list(ASSETS):
    raise ValueError(f"Legacy targets must have exactly columns {list(ASSETS)}")
target_rows = targets.loc[targets.index.isin(loadings.index)].copy()
if len(target_rows) != len(loadings) or target_rows.isna().any(axis=None):
    raise ValueError("Legacy targets must contain one complete target row per rebalance")
if not np.isfinite(target_rows.to_numpy(dtype=float)).all() or not np.allclose(target_rows.sum(axis=1), 1.0, atol=1e-10):
    raise ValueError("Legacy target rows must be finite and fully invested")
if list(equity.columns) != ["value"] or not np.isfinite(equity["value"].to_numpy(dtype=float)).all() or (equity["value"] <= 0).any():
    raise ValueError("Legacy equity must contain one finite, positive value column")
equity_rows = []
for date in loadings.index:
    equity_observation_date, equity_value = equity_observation_on_or_after(equity, date)
    equity_rows.append({
        "rebalance_date": date,
        "equity_observation_date": equity_observation_date,
        "equity_observation_lag_days": int((equity_observation_date - date).days),
        "persisted_equity_value": equity_value,
    })
equity_by_rebalance = pd.DataFrame(equity_rows).set_index("rebalance_date")
if (equity_by_rebalance["equity_observation_lag_days"] < 0).any() or (equity_by_rebalance["equity_observation_lag_days"] > 3).any():
    raise ValueError("Legacy equity observation must occur on or within three days after each rebalance")

if header.get("n_rebalances", {}).get("total") != 126 or header.get("window", {}).get("first_rebalance") != "2016-01-04" or header.get("window", {}).get("last_rebalance") != "2026-06-01":
    raise ValueError("Legacy header does not match expected 10-year rebalance coverage")
if list(macro.columns) != ["cpi_yoy", "t10y2y", "hy_oas", *MACRO_COLUMNS]:
    raise ValueError("Macro panel schema changed")
if macro.index.tz is not None or not macro.index.is_monotonic_increasing or macro.index.has_duplicates:
    raise ValueError("Macro panel index must be timezone-naive, increasing, and unique")

macro_rows = []
for date in loadings.index:
    macro_row = prior_macro_row(macro, date)
    macro_source_date = pd.Timestamp(macro_row.name)
    macro_rows.append({
        "rebalance_date": date,
        "macro_source_date": macro_source_date,
        "macro_source_lag_days": int((date - macro_source_date).days),
        **{field: float(macro_row[field]) for field in MACRO_COLUMNS},
    })
macro_by_date = pd.DataFrame(macro_rows).set_index("rebalance_date")
if (macro_by_date["macro_source_lag_days"] <= 0).any():
    raise ValueError("Macro source date must precede every rebalance")

log_crosscheck = []
for date, row in loadings.iterrows():
    key = str(date)
    log_parse_ok = bool(decision_log["parse_ok"][key])
    if bool(row["parse_ok"]) != log_parse_ok:
        raise ValueError(f"Loading/decision-log parse disagreement at {date.date()}")
    if not bool(row["parse_ok"]):
        log_crosscheck.append({"rebalance_date": date, "decision_log_loading_match": False, "decision_log_mismatch_reason": "parse_failure"})
        continue
    mismatched_axes = [axis for axis in AXES if abs(float(row[axis]) - float(decision_log["loadings"][key][axis])) > 1e-12]
    log_crosscheck.append({
        "rebalance_date": date,
        "decision_log_loading_match": not mismatched_axes,
        "decision_log_mismatch_reason": ", ".join(mismatched_axes) if mismatched_axes else "",
    })
log_crosscheck = pd.DataFrame(log_crosscheck).set_index("rebalance_date")
known_mismatch_dates = pd.DatetimeIndex(["2025-10-01", "2026-04-01", "2026-05-01"])
observed_mismatch_dates = log_crosscheck.index[(~log_crosscheck["decision_log_loading_match"]) & (log_crosscheck["decision_log_mismatch_reason"] != "parse_failure")]
if not observed_mismatch_dates.equals(known_mismatch_dates):
    raise ValueError(f"Legacy loading/decision-log mismatch dates changed: {list(observed_mismatch_dates)}")

validated_source_inventory = pd.DataFrame([
    {"role": name, "file": repo_relative_path(REPO_ROOT, path, name), "sha256": source_hashes_before[name]}
    for name, path in source_paths.items()
])
display(Markdown(markdown_lines([
    "**Legacy source validation passed (hash-pinned, not manifest-backed).**",
    f"Rebalance coverage: `{loadings.index.min().date()}` to `{loadings.index.max().date()}` (`{len(loadings)}` monthly records).",
    "The source status is explicitly **non-manifested legacy/reconstructed**; it must not be interpreted as a completed `factor_run.v1` bundle.",
])))
display(validated_source_inventory)

In [ ]:
behavior = loadings.join(macro_by_date).join(equity_by_rebalance).join(log_crosscheck)
behavior["p_memorized"] = [
    float(decision_log["p_memorized"][str(date)]) if decision_log["p_memorized"][str(date)] is not None else np.nan
    for date in behavior.index
]
behavior["steered"] = [bool(decision_log["steered"][str(date)]) for date in behavior.index]
behavior["conviction"] = [
    float(decision_log["conviction"][str(date)]) if decision_log["conviction"][str(date)] is not None else np.nan
    for date in behavior.index
]
if behavior.loc[behavior["parse_ok"], ["p_memorized", "conviction"]].isna().any(axis=None):
    raise ValueError("Parsed loading rows require finite recall score and conviction")
if behavior.loc[~behavior["parse_ok"], ["p_memorized", "conviction"]].notna().any(axis=None):
    raise ValueError("Failed parse row must preserve unavailable recall score and conviction")
for axis in AXES:
    behavior[f"expanding_z_{axis}"] = expanding_prior_zscore(behavior[axis])

long_rows = []
for date, row in behavior.iterrows():
    for asset in ASSETS:
        pseudo = ASSET_KEYS[asset]
        category = ASSET_CATEGORIES[asset]
        record = {
            "rebalance_date": date,
            "macro_source_date": row["macro_source_date"],
            "macro_source_lag_days": int(row["macro_source_lag_days"]),
            "equity_observation_date": row["equity_observation_date"],
            "equity_observation_lag_days": int(row["equity_observation_lag_days"]),
            "segment": row["segment"],
            "parse_ok": bool(row["parse_ok"]),
            "asset_id": pseudo,
            "ticker": asset,
            "category": category,
            "p_memorized": float(row["p_memorized"]) if pd.notna(row["p_memorized"]) else np.nan,
            "steered": bool(row["steered"]),
            "conviction": float(row["conviction"]) if pd.notna(row["conviction"]) else np.nan,
            "decision_log_loading_match": bool(row["decision_log_loading_match"]),
            "decision_log_mismatch_reason": row["decision_log_mismatch_reason"],
            "persisted_target_weight": float(target_rows.loc[date, asset]),
            "persisted_equity_value": float(row["persisted_equity_value"]),
            **{field: float(row[field]) for field in MACRO_COLUMNS},
            **{axis: (float(row[axis]) if pd.notna(row[axis]) else np.nan) for axis in AXES},
            **{f"expanding_z_{axis}": (float(row[f"expanding_z_{axis}"]) if pd.notna(row[f"expanding_z_{axis}"]) else np.nan) for axis in AXES},
        }
        if bool(row["parse_ok"]):
            raw_tilt = sum(float(row[axis]) * EXPOSURE_MAP[category][axis] for axis in AXES)
            guarded_tilt = raw_tilt * (1.0 - float(row["p_memorized"]))
            q_value = guarded_tilt * float(np.clip(row["conviction"], 0.0, 1.0)) / 252.0
        else:
            raw_tilt, guarded_tilt, q_value = np.nan, np.nan, np.nan
        record.update({"raw_tilt": raw_tilt, "guarded_tilt": guarded_tilt, "bl_q": q_value})
        long_rows.append(record)
behavior_panel = pd.DataFrame(long_rows).sort_values(["rebalance_date", "asset_id"]).reset_index(drop=True)
if len(behavior_panel) != len(loadings) * len(ASSETS):
    raise ValueError("Appendix H panel must contain 126 dates × four fixed assets")
if behavior_panel.loc[behavior_panel["rebalance_date"] == pd.Timestamp("2024-03-01"), ["raw_tilt", "guarded_tilt", "bl_q"]].notna().any(axis=None):
    raise ValueError("Failed parse must remain unavailable in derived factor fields")
if behavior_panel.loc[behavior_panel["rebalance_date"] != pd.Timestamp("2024-03-01"), ["raw_tilt", "guarded_tilt", "bl_q"]].isna().any(axis=None):
    raise ValueError("Parsed records require finite deterministic transformations")

PANEL_NAME = "appendix_h_deterministic_factor_panel.parquet"
PANEL_SCHEMA = "appendix_h.deterministic_factor_panel.v1"
panel_path = OUTPUT_DIR / PANEL_NAME
behavior_panel.to_parquet(panel_path, index=False)

asset_crosswalk = pd.DataFrame([
    {"Pseudo asset": ASSET_KEYS[asset], "Reporting ticker": asset, "Category": ASSET_CATEGORIES[asset]}
    for asset in ASSETS
])
display(Markdown("## Recorded factor behavior and deterministic transmission"))
display(Markdown(
    "The loadings table is the display authority. The decision log provides `p_memorized`, `steered`, and `conviction`, "
    "plus a cross-check. The three disclosed loading/log mismatches are not silently reconciled. "
    "Raw tilt, guarded tilt, and Q are a deterministic display reconstruction—not a rerun of the BL optimizer or final allocation."
))
display(asset_crosswalk)
display(pd.DataFrame({
    "Observed": [len(loadings), int(loadings["parse_ok"].sum()), len(observed_mismatch_dates), int((equity_by_rebalance["equity_observation_lag_days"] > 0).sum())],
}, index=["Monthly records", "Parsed loading records", "Loading/log mismatch dates", "Non-trading rebalances carried to next equity observation"]))

In [ ]:
macro_matrix = macro_by_date.loc[:, list(MACRO_COLUMNS)].T
fig, ax = plt.subplots(figsize=(15.2, 4.7), constrained_layout=True)
image = ax.imshow(macro_matrix.to_numpy(dtype=float), aspect="auto", interpolation="nearest", cmap=DIVERGING_CMAP, vmin=-3.0, vmax=3.0)
ax.set_yticks(range(len(MACRO_COLUMNS)), labels=["Inflation (CPI YoY)", "Yield curve (10Y−2Y)", "High-yield spread"])
step = max(1, len(macro_matrix.columns) // 13)
ticks = np.arange(0, len(macro_matrix.columns), step)
ax.set_xticks(ticks, labels=[macro_matrix.columns[i].strftime("%Y-%m") for i in ticks], rotation=45, ha="right")
ax.set_xlabel("Factor rebalance date")
ax.set_title("Strict-as-of macro z-scores feeding the historical factor record", loc="left", fontweight="semibold")
bar = fig.colorbar(image, ax=ax, fraction=0.025, pad=0.02)
bar.set_label("Producer-supplied macro z-score (0 = historical center)")
fig.savefig(OUTPUT_DIR / "appendix_h_pit_macro_zscores.png", dpi=180, bbox_inches="tight", facecolor=SURFACE)
plt.show()
display(Markdown("**Print/CVD-safe table:** exact signed macro z-scores and the prior source-date lag used at every rebalance."))
display(macro_by_date[["macro_source_date", "macro_source_lag_days", *MACRO_COLUMNS]])

In [ ]:
loading_z_columns = [f"expanding_z_{axis}" for axis in AXES]
loading_z_matrix = behavior.loc[:, loading_z_columns].rename(columns=lambda field: field.removeprefix("expanding_z_")).T
masked = np.ma.masked_invalid(loading_z_matrix.to_numpy(dtype=float))
fig, ax = plt.subplots(figsize=(15.2, 5.8), constrained_layout=True)
image = ax.imshow(masked, aspect="auto", interpolation="nearest", cmap=DIVERGING_CMAP, vmin=-3.0, vmax=3.0)
ax.set_yticks(range(len(AXES)), labels=["Inflation", "Growth", "Credit stress", "Policy", "Risk appetite"])
step = max(1, len(loading_z_matrix.columns) // 13)
ticks = np.arange(0, len(loading_z_matrix.columns), step)
ax.set_xticks(ticks, labels=[loading_z_matrix.columns[i].strftime("%Y-%m") for i in ticks], rotation=45, ha="right")
ax.set_xlabel("Rebalance date")
ax.set_title("Recorded AI loading oscillation: expanding decision-time z-scores", loc="left", fontweight="semibold")
bar = fig.colorbar(image, ax=ax, fraction=0.025, pad=0.02)
bar.set_label("Loading z-score versus prior valid recorded history")
parse_failure_column = int(np.flatnonzero(loading_z_matrix.columns == pd.Timestamp("2024-03-01"))[0])
ax.axvspan(parse_failure_column - 0.5, parse_failure_column + 0.5, facecolor="none", edgecolor=INK, hatch="///", linewidth=0.0, label="2024-03 parse failure")
ax.legend(loc="upper right", fontsize=7.5)
fig.savefig(OUTPUT_DIR / "appendix_h_factor_loading_zscores.png", dpi=180, bbox_inches="tight", facecolor=SURFACE)
plt.show()
display(Markdown(
    f"**Standardization contract:** each value uses only prior valid loading observations for the same axis; "
    f"minimum prior history = {LOADING_Z_MIN_HISTORY} months; sample standard deviation (`ddof={LOADING_Z_DDOF}`). "
    "The early warm-up and the failed March 2024 parse are unavailable, not zero."
))
display(behavior[[*AXES, *loading_z_columns, "parse_ok"]])

In [ ]:
tilt_fields = ("raw_tilt", "guarded_tilt", "bl_q")
fig, axes_grid = plt.subplots(len(ASSETS), len(tilt_fields), figsize=(16, 10.5), sharex=True, constrained_layout=True)
for row_idx, asset in enumerate(ASSETS):
    rows = behavior_panel.loc[behavior_panel["ticker"] == asset].sort_values("rebalance_date")
    for col_idx, field in enumerate(tilt_fields):
        axis = axes_grid[row_idx, col_idx]
        values = rows[field].to_numpy(dtype=float)
        limit = float(np.nanmax(np.abs(values)))
        if not np.isfinite(limit) or limit <= 0:
            raise ValueError(f"{asset} {field} requires finite signed values")
        axis.plot(rows["rebalance_date"], values, color=ASSET_COLORS[asset], marker=ASSET_MARKERS[asset], markevery=max(1, len(rows) // 12), markersize=3.8)
        axis.axhline(0.0, color=SECONDARY, linewidth=0.8)
        axis.set_ylim(-limit * 1.08, limit * 1.08)
        axis.grid(axis="y")
        if row_idx == 0:
            axis.set_title({"raw_tilt": "Raw unitless tilt", "guarded_tilt": "Recall-adjusted unitless tilt", "bl_q": "BL Q input"}[field], loc="left", fontweight="semibold")
        if col_idx == 0:
            axis.set_ylabel(f"{ASSET_KEYS[asset]}\n{asset}")
        if row_idx == len(ASSETS) - 1:
            axis.set_xlabel("Rebalance date")
fig.suptitle("Deterministic factor transmission: loading → raw tilt → recall-adjusted tilt → Q", fontsize=13, fontweight="semibold")
fig.savefig(OUTPUT_DIR / "appendix_h_signed_tilts_bl_q.png", dpi=180, bbox_inches="tight", facecolor=SURFACE)
plt.show()
display(Markdown("**Print/CVD-safe table:** exact signed values shown above. Positive and negative signs are meaningful; magnitudes use separate axes because tilt and Q have different units/scales."))
display(behavior_panel[["rebalance_date", "asset_id", "ticker", "raw_tilt", "guarded_tilt", "bl_q", "p_memorized", "conviction"]])

In [ ]:
fig, axes_grid = plt.subplots(2, 2, figsize=(13.5, 7.8), sharex=True, sharey=True, constrained_layout=True)
for axis, asset in zip(axes_grid.flat, ASSETS, strict=True):
    rows = behavior_panel.loc[behavior_panel["ticker"] == asset].sort_values("rebalance_date")
    axis.plot(rows["rebalance_date"], rows["persisted_target_weight"], color=ASSET_COLORS[asset], marker=ASSET_MARKERS[asset], markevery=max(1, len(rows) // 12), markersize=4)
    axis.set_ylim(0, 1)
    axis.grid(axis="y")
    axis.set_title(f"{ASSET_KEYS[asset]} — {asset}\n{ASSET_CATEGORIES[asset].replace('_', ' ')}", loc="left", fontweight="semibold")
    axis.set_ylabel("Persisted target weight")
    axis.set_xlabel("Rebalance date")
fig.suptitle("Observed target weights: portfolio output preserved from the legacy stream", fontsize=13, fontweight="semibold")
fig.savefig(OUTPUT_DIR / "appendix_h_persisted_target_weights.png", dpi=180, bbox_inches="tight", facecolor=SURFACE)
plt.show()
display(Markdown(
    "These are persisted final targets. The legacy stream does not persist a standalone HRP baseline, BL-only allocation, or validated final-minus-HRP performance path. "
    "The appendix therefore presents no invented allocation attribution or AI-versus-HRP performance result."
))
display(target_rows.rename(columns={asset: f"{ASSET_KEYS[asset]} ({asset})" for asset in ASSETS}))

In [ ]:
active_equity = equity.loc[equity.index >= loadings.index.min(), "value"].copy()
fig, ax = plt.subplots(figsize=(13.5, 4.8), constrained_layout=True)
ax.plot(active_equity.index, active_equity, color="#2a78d6", linewidth=1.8)
ax.grid(axis="y")
ax.set_title("Persisted historical equity level from the first factor rebalance", loc="left", fontweight="semibold")
ax.set_xlabel("Date")
ax.set_ylabel("Persisted portfolio value")
ax.text(0.01, 0.03, "Level only. No local return, drawdown, CAGR, Sharpe, or attribution calculation is performed.", transform=ax.transAxes, fontsize=8, color=SECONDARY, bbox={"facecolor": SURFACE, "edgecolor": GRID, "pad": 3})
fig.savefig(OUTPUT_DIR / "appendix_h_persisted_equity_level.png", dpi=180, bbox_inches="tight", facecolor=SURFACE)
plt.show()

In [ ]:
macro_extremeness = np.sqrt((macro_by_date.loc[:, list(MACRO_COLUMNS)] ** 2).sum(axis=1))
most_unusual_date = pd.Timestamp(macro_extremeness.idxmax())
nearest_neutral_date = pd.Timestamp(macro_extremeness.idxmin())
if most_unusual_date == nearest_neutral_date:
    raise ValueError("Worked trace selectors must identify distinct macro-state dates")
role_by_date = {most_unusual_date: "Largest macro-state magnitude", nearest_neutral_date: "Nearest macro-state to neutral"}
worked = behavior_panel.loc[behavior_panel["rebalance_date"].isin(role_by_date)].copy()
worked["selection_reason"] = worked["rebalance_date"].map(role_by_date)
worked_columns = [
    "selection_reason", "rebalance_date", "macro_source_date", "macro_source_lag_days", "asset_id", "ticker", "category",
    *MACRO_COLUMNS, *AXES, *loading_z_columns, "parse_ok", "p_memorized", "conviction", "raw_tilt", "guarded_tilt", "bl_q",
    "persisted_target_weight", "decision_log_loading_match", "decision_log_mismatch_reason",
]
worked = worked[worked_columns].sort_values(["rebalance_date", "asset_id"]).reset_index(drop=True)
display(Markdown("## Worked traces selected from macro state, not performance"))
display(Markdown("The two dates are chosen using only the strict-as-of macro-state magnitude. This deliberately avoids selecting examples by return or portfolio outcome."))
display(worked)

worked_factor_labels = {
    "macro_source_date": "Macro date",
    "macro_source_lag_days": "Macro lag (d)",
    "cpi_yoy_z": "CPI z",
    "t10y2y_z": "Curve z",
    "hy_oas_z": "HY z",
    "inflation": "L: inflation",
    "growth": "L: growth",
    "credit_stress": "L: credit",
    "policy": "L: policy",
    "risk_appetite": "L: risk",
    "expanding_z_inflation": "z: inflation",
    "expanding_z_growth": "z: growth",
    "expanding_z_credit_stress": "z: credit",
    "expanding_z_policy": "z: policy",
    "expanding_z_risk_appetite": "z: risk",
    "parse_ok": "Parsed",
    "p_memorized": "Recall p",
    "conviction": "Conviction",
    "decision_log_loading_match": "Log match",
    "decision_log_mismatch_reason": "Log mismatch",
}
fig, axes_grid = plt.subplots(4, 1, figsize=(18, 11.2), constrained_layout=True, gridspec_kw={"height_ratios": [1.0, 1.45, 1.0, 1.45]})
for macro_axis, asset_axis, date in zip(axes_grid[::2], axes_grid[1::2], (most_unusual_date, nearest_neutral_date), strict=True):
    factor_row = behavior.loc[date]
    factor_fields = ["macro_source_date", "macro_source_lag_days", *MACRO_COLUMNS, *AXES, *loading_z_columns, "parse_ok", "p_memorized", "conviction", "decision_log_loading_match", "decision_log_mismatch_reason"]
    factor_table = pd.DataFrame([{field: factor_row[field] for field in factor_fields}]).rename(columns=worked_factor_labels)
    factor_formatted = factor_table.apply(lambda column: column.map(lambda value: "" if pd.isna(value) else f"{value:.4f}" if isinstance(value, (float, np.floating)) else str(value)))
    macro_axis.axis("off")
    macro_axis.set_title(f"{role_by_date[date]}: {date.date()} | strict-as-of macro state and recorded factor loadings", loc="left", fontweight="semibold", pad=7)
    macro_table = macro_axis.table(cellText=factor_formatted.values, colLabels=factor_formatted.columns, loc="center", cellLoc="left", colLoc="left")
    macro_table.auto_set_font_size(False)
    macro_table.set_fontsize(5.8)
    macro_table.scale(1.0, 1.35)
    for (row_index, _), cell in macro_table.get_celld().items():
        cell.set_edgecolor(GRID)
        cell.set_linewidth(0.45)
        cell.set_facecolor("#f0efec" if row_index == 0 else SURFACE)
        cell.get_text().set_color(INK if row_index == 0 else SECONDARY)

    shown = worked.loc[worked["rebalance_date"] == date].copy()
    shown = shown[["asset_id", "ticker", "category", "p_memorized", "conviction", "raw_tilt", "guarded_tilt", "bl_q", "persisted_target_weight", "decision_log_loading_match", "decision_log_mismatch_reason"]]
    formatted = shown.apply(lambda column: column.map(lambda value: "" if pd.isna(value) else f"{value:.5f}" if isinstance(value, (float, np.floating)) else str(value)))
    asset_axis.axis("off")
    asset_axis.set_title("Fixed exposure map → raw tilt → recall discount → BL Q → persisted final target", loc="left", fontweight="semibold", pad=7)
    asset_table = asset_axis.table(cellText=formatted.values, colLabels=formatted.columns, loc="center", cellLoc="left", colLoc="left")
    asset_table.auto_set_font_size(False)
    asset_table.set_fontsize(6.4)
    asset_table.scale(1.0, 1.35)
    for (row_index, _), cell in asset_table.get_celld().items():
        cell.set_edgecolor(GRID)
        cell.set_linewidth(0.45)
        cell.set_facecolor("#f0efec" if row_index == 0 else SURFACE)
        cell.get_text().set_color(INK if row_index == 0 else SECONDARY)
fig.savefig(OUTPUT_DIR / "appendix_h_worked_factor_trace.png", dpi=180, bbox_inches="tight", facecolor=SURFACE)
plt.show()

## What this evidence supports—and what it does not

**Supported:** the legacy 10-year record lets us inspect the observed movement of the five AI regime loadings, the fixed deterministic mapping into unitless tilts and BL `Q`, the recall attenuation inputs, and the persisted final target/equity outputs.

**Not supported:** the stream does not provide an immutable completed Factor-run contract, historical macro release vintages, an HRP-only counterfactual, a BL-only counterfactual, or a sealed future performance evaluation. Accordingly, the charts are not evidence of causal attribution, predictive alpha, or a validated performance advantage versus HRP.

The deterministic part of this appendix is deliberately narrow: it captures the *published transformation* from recorded loading to tilt/Q. It does not claim to reproduce the original model's internal reasoning or to reconstruct every historical allocation decision.

## Future work — RIDRA

RIDRA is **not implemented in Appendix H**. This appendix imports no RIDRA code, fits no rule model, runs no ADWIN drift detector, expands no rule, writes no candidate allocation, and does not backtest a surrogate.

A future RIDRA study should be treated as a separate, governed research specification—not a relabeling of the legacy historical factor. Before an adaptive rule can influence BL `Q` or target weights, the study must freeze its rule grammar, categorical target, detector input, `delta`, aging/decay, support/pruning policy, and no-rule fallback. It must use data that are available at each decision time, preserve an immutable chronological ledger, predict/lock an action before observing its outcome, and perform any update only after the next completed observation. Drift should produce a versioned alert or shadow evaluation, not an automatic allocation change. Promotion requires independent validation, approved guardrails, and a rollback path.

In [ ]:
FIGURES = (
    "appendix_h_pit_macro_zscores.png",
    "appendix_h_factor_loading_zscores.png",
    "appendix_h_signed_tilts_bl_q.png",
    "appendix_h_persisted_target_weights.png",
    "appendix_h_persisted_equity_level.png",
    "appendix_h_worked_factor_trace.png",
)

existing_manifest = OUTPUT_DIR / "presentation_manifest.json"
if existing_manifest.is_file():
    previous = json.loads(existing_manifest.read_text(encoding="utf-8"))
    previous_hashes = previous.get("source_hashes")
    if previous_hashes is not None and previous_hashes != source_hashes_before:
        raise RuntimeError("Refusing to overwrite Appendix H outputs derived from different source hashes")

assert_sources_unchanged(source_paths, source_hashes_before)
panel_entry = {
    "file": PANEL_NAME,
    "schema": PANEL_SCHEMA,
    "sha256": sha256_file(panel_path),
    "rows": len(behavior_panel),
    "columns": list(behavior_panel.columns),
}
payload = {
    "schema": "appendix_h_deterministic_factor.presentation.v1",
    "appendix_id": "appendix_h_deterministic_factor",
    "completed": True,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "presentation_only": True,
    "notebook": {"file": repo_relative_path(REPO_ROOT, NOTEBOOK_PATH, "notebook"), "sha256": sha256_file(NOTEBOOK_PATH)},
    "source_status": SOURCE_STATUS,
    "source_hashes": source_hashes_before,
    "source_inventory": {
        name: {"file": repo_relative_path(REPO_ROOT, path, name), "sha256": source_hashes_before[name]}
        for name, path in source_paths.items()
    },
    "macro_asof_contract": {
        "rule": "macro_source_date < rebalance_date",
        "zscore_columns": list(MACRO_COLUMNS),
        "release_vintage_pit": False,
        "late_source_lag_days_max": int(macro_by_date["macro_source_lag_days"].max()),
    },
    "loading_standardization": {
        "method": "expanding_prior_history",
        "axes": list(AXES),
        "minimum_prior_valid_observations": LOADING_Z_MIN_HISTORY,
        "ddof": LOADING_Z_DDOF,
        "parse_failure_policy": "unavailable_not_zero",
    },
    "deterministic_transformation": {
        "factor_module": "macro_framework.factor_scoring",
        "exposure_map": EXPOSURE_MAP,
        "raw_tilt": "sum(loading_axis * category_exposure_axis)",
        "guarded_tilt": "raw_tilt * (1 - p_memorized)",
        "bl_q": "guarded_tilt * clip(conviction, 0, 1) / 252",
        "not_allocation_replay": True,
    },
    "decision_log_crosscheck": {
        "loading_table_display_authority": True,
        "parse_failure_date": "2024-03-01",
        "mismatch_dates": [date.strftime("%Y-%m-%d") for date in observed_mismatch_dates],
    },
    "panel": panel_entry,
    "outputs": output_inventory(OUTPUT_DIR, FIGURES),
}
manifest_path = OUTPUT_DIR / "presentation_manifest.json"
manifest_path.write_text(json.dumps(payload, indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8")
assert_sources_unchanged(source_paths, source_hashes_before)
display(Markdown(markdown_lines([
    f"**Appendix H presentation manifest:** `{manifest_path}`",
    f"SHA-256: `{sha256_file(manifest_path)}`",
    "Source inputs were hash-verified before and after rendering.",
])))